# 04 - Modelling

## Data sources
- **ML-HydPARK** (Zenodo, v0.0.5): cleaned experimental thermodynamic data for metal hydrides (~770 entries)
- **ElementalH_Ef** and **ElementalH_Ef_MP**: reserved for compositional feature engineering (future step)

## Target Variables
- `Temperature_oC`
- `Hydrogen_Weight_Percent`

## Goals
- Investigate baseline models for each dataset
- Evaluation and fine tuning of chosen models
- SHAP analysis for model interpretability


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
from xgboost import XGBRegressor
import shap

In [5]:
# Loading dataset A
X_A_train = pd.read_csv('../data/processed/splits/X_A_train.csv')
X_A_val = pd.read_csv('../data/processed/splits/X_A_val.csv')
X_A_test = pd.read_csv('../data/processed/splits/X_A_test.csv')
y_A_train = pd.read_csv('../data/processed/splits/y_A_train.csv').squeeze()
y_A_val = pd.read_csv('../data/processed/splits/y_A_val.csv').squeeze()
y_A_test = pd.read_csv('../data/processed/splits/y_A_test.csv').squeeze()

# Loading dataset B
X_B_train = pd.read_csv('../data/processed/splits/X_B_train.csv')
X_B_val = pd.read_csv('../data/processed/splits/X_B_val.csv')
X_B_test = pd.read_csv('../data/processed/splits/X_B_test.csv')
y_B_train = pd.read_csv('../data/processed/splits/y_B_train.csv').squeeze()
y_B_val = pd.read_csv('../data/processed/splits/y_B_val.csv').squeeze()
y_B_test = pd.read_csv('../data/processed/splits/y_B_test.csv').squeeze()

print(X_A_train.shape, X_A_val.shape, X_A_test.shape)
print(X_B_train.shape, X_B_val.shape, X_B_test.shape)


(518, 11) (111, 11) (112, 11)
(266, 10) (57, 10) (58, 10)


## Dataset A

Target variable: `Hydrogen_Weight_Percent`

### Linear Regression: baseline model

In [3]:
# Model instantiation, training and validation
lr_A = LinearRegression()
lr_A.fit(X_A_train, y_A_train)
y_A_val_pred_lr = lr_A.predict(X_A_val)


# Evaluation
print('R²:', r2_score(y_A_val, y_A_val_pred_lr))
print('RMSE:', np.sqrt(mean_squared_error(y_A_val, y_A_val_pred_lr)))
print('MAE:', mean_absolute_error(y_A_val, y_A_val_pred_lr))

R²: 0.29595540128489306
RMSE: 0.5387057686587513
MAE: 0.3392752642821463


### Random Forest: baseline model

In [6]:
# Model instantiation, training and validation
rf_A = RandomForestRegressor()
rf_A.fit(X_A_train, y_A_train)
y_A_val_pred_rf = rf_A.predict(X_A_val)


# Evaluation
print('R²:', r2_score(y_A_val, y_A_val_pred_rf))
print('RMSE:', np.sqrt(mean_squared_error(y_A_val, y_A_val_pred_rf)))
print('MAE:', mean_absolute_error(y_A_val, y_A_val_pred_rf))

R²: 0.4940697613785947
RMSE: 0.45666408713722184
MAE: 0.31015917761497236


### XGBoost Regressor: baseline model

In [ ]:
# Model instantiation, training and validation
xgb_A = XGBRegressor()
xgb_A.fit(X_A_train, y_A_train)
y_A_val_pred_xgb = xgb_A.predict(X_A_val)


# Evaluation
print('R²:', r2_score(y_A_val, y_A_val_pred_xgb))
print('RMSE:', np.sqrt(mean_squared_error(y_A_val, y_A_val_pred_xgb)))
print('MAE:', mean_absolute_error(y_A_val, y_A_val_pred_xgb))

R²: 0.4207742780016551
RMSE: 0.48862474909310166
MAE: 0.3333662903103805


## Baseline Models Comparison

Random Forest is the best performing model compared to XGBoost and Linear Regression:

| Model | R² | RMSE | MAE |
|---|---|---|---|
| Linear Regression | 0.296 | 0.539 | 0.339 |
| Random Forest | 0.495 | 0.456 | 0.310 |
| XGBoost | 0.421 | 0.489 | 0.333 |

Random Forest is the best model with an R²=0.495, which means that the vanilla model explains approximately half of the variance in `Hydrogen_Weight_Percent`.

## Next Steps

Feature engineering: adding the `ElementalH_Ef` to the features could help the model better understand and distinguish between hydrogen storage materials with similar enthalpy of formation but different elemental composition.